<a href="https://colab.research.google.com/github/hectorjimenez12/CC5205_Proyecto/blob/main/Proyecto_Mineria_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
#library
import csv
import datetime as dt
import json
import os
import statistics
import time


import numpy as np
import pandas as pd
import requests
pd.set_option('display.max_columns', 50)

#https://andrew-muller.medium.com/scraping-steam-user-reviews-9a43f9e38c92

Useful functions for downloading data.

In [16]:
def get_reviews(appid, params={'json':1}):
        url = 'https://store.steampowered.com/appreviews/'
        response = requests.get(url=url+appid, params=params, headers={'User-Agent': 'Mozilla/5.0'})
        return response.json()

params = {'json':1}
response = get_reviews('413150', params)
cursor = response['cursor']
params['cursor'] = cursor.encode()
response_2 = get_reviews('413150', params)
print(response)
print(response_2)

{'success': 1, 'query_summary': {'num_reviews': 20, 'review_score': 9, 'review_score_desc': 'Overwhelmingly Positive', 'total_positive': 257315, 'total_negative': 2930, 'total_reviews': 260245}, 'reviews': [{'recommendationid': '143140155', 'author': {'steamid': '76561199096033970', 'num_games_owned': 53, 'num_reviews': 1, 'playtime_forever': 62525, 'playtime_last_two_weeks': 0, 'playtime_at_review': 60975, 'last_played': 1691479002}, 'language': 'english', 'review': "personally i found this game really enjoyable but i don't think I've played enough to give a proper review just yet", 'timestamp_created': 1690804329, 'timestamp_updated': 1690804329, 'voted_up': True, 'votes_up': 611, 'votes_funny': 865, 'weighted_vote_score': '0.951825141906738281', 'comment_count': 0, 'steam_purchase': True, 'received_for_free': False, 'written_during_early_access': False, 'hidden_in_steam_china': True, 'steam_china_location': ''}, {'recommendationid': '143279482', 'author': {'steamid': '76561198400835

{'success': 1, 'query_summary': {'num_reviews': 20, 'review_score': 9, 'review_score_desc': 'Overwhelmingly Positive', 'total_positive': 257315, 'total_negative': 2930, 'total_reviews': 260245}, 'reviews': [{'recommendationid': '143140155', 'author': {'steamid': '76561199096033970', 'num_games_owned': 53, 'num_reviews': 1, 'playtime_forever': 62525, 'playtime_last_two_weeks': 0, 'playtime_at_review': 60975, 'last_played': 1691479002}, 'language': 'english', 'review': "personally i found this game really enjoyable but i don't think I've played enough to give a proper review just yet", 'timestamp_created': 1690804329, 'timestamp_updated': 1690804329, 'voted_up': True, 'votes_up': 606, 'votes_funny': 863, 'weighted_vote_score': '0.953347444534301758', 'comment_count': 0, 'steam_purchase': True, 'received_for_free': False, 'written_during_early_access': False, 'hidden_in_steam_china': True, 'steam_china_location': ''}, {'recommendationid': '143279482', 'author': {'steamid': '76561198400835

In [58]:
steam_ids = json_data['applist']['apps']
df_steam = pd.DataFrame.from_dict(steam_ids, orient='columns')
print(df_steam.shape)

#filt
tf_demo = np.array(list(map( lambda x: True if 'demo' in x in x else False ,df_steam.name.values)))
tf_test = np.array(list(map( lambda x: True if 'test' in x in x else False ,df_steam.name.values)))
tf_empty =  (df_steam.name.values == '') 
df_filt = df_steam[ ~ ( tf_test|tf_demo | tf_empty) ]
print(df_filt.head())
df_filt.shape


(172749, 2)
      appid                             name
36  2225790                       Space Draw
37  2225800  3х9 Царство: Дорога приключений
38  2225810       Alchemia Story Sound Track
39  2225830                      Cash Cow DX
40  2225860                   PESTICIDE Demo


(169492, 2)

In [59]:
# generate sorted app_list from steamspy data
app_list = df_filt[['appid', 'name']].sort_values('appid').reset_index(drop=True)
#app_list.to_csv(dir_save_files + 'app_list.csv', index=False)
#app_list = pd.read_csv(dir_save_files + 'app_list.csv')
app_list.head()

,appid,name
0,5,Dedicated Server
1,7,Steam Client
2,8,winui2
3,10,Counter-Strike
4,20,Team Fortress Classic


In [63]:
def get_app_data(start, stop, parser, pause):
    """Return list of app data generated from parser.

    parser : function to handle request
    """
    app_data = []

    # iterate through each row of app_list, confined by start and stop
    for index, row in app_list[start:stop].iterrows():
        print('Current index: {}'.format(index), end='\r')

        appid = row['appid']
        name = row['name']

        # retrive app data for a row, handled by supplied parser, and append to list
        data = parser(appid, name)
        app_data.append(data)

        time.sleep(pause) # prevent overloading api with requests

    return app_data


def process_batches(parser, app_list, download_path, data_filename, index_filename,
                    columns, begin=0, end=-1, batchsize=100, pause=1):
    """Process app data in batches, writing directly to file.

    parser : custom function to format request
    app_list : dataframe of appid and name
    download_path : path to store data
    data_filename : filename to save app data
    index_filename : filename to store highest index written
    columns : column names for file

    Keyword arguments:

    begin : starting index (get from index_filename, default 0)
    end : index to finish (defaults to end of app_list)
    batchsize : number of apps to write in each batch (default 100)
    pause : time to wait after each api request (defualt 1)

    returns: none
    """
    print('Starting at index {}:\n'.format(begin))

    # by default, process all apps in app_list
    if end == -1:
        end = len(app_list) + 1

    # generate array of batch begin and end points
    batches = np.arange(begin, end, batchsize)
    batches = np.append(batches, end)

    apps_written = 0
    batch_times = []

    for i in range(len(batches) - 1):
        start_time = time.time()

        start = batches[i]
        stop = batches[i+1]

        app_data = get_app_data(start, stop, parser, pause)

        rel_path = os.path.join(download_path, data_filename)

        # writing app data to file
        with open(rel_path, 'a', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=columns, extrasaction='ignore')

            for j in range(3,0,-1):
                print("\rAbout to write data, don't stop script! ({})".format(j), end='')
                time.sleep(0.5)

            writer.writerows(app_data)
            print('\rExported lines {}-{} to {}.'.format(start, stop-1, data_filename), end=' ')

        apps_written += len(app_data)

        idx_path = os.path.join(download_path, index_filename)

        # writing last index to file
        with open(idx_path, 'w') as f:
            index = stop
            print(index, file=f)

        # logging time taken
        end_time = time.time()
        time_taken = end_time - start_time

        batch_times.append(time_taken)
        mean_time = statistics.mean(batch_times)

        est_remaining = (len(batches) - i - 2) * mean_time

        remaining_td = dt.timedelta(seconds=round(est_remaining))
        time_td = dt.timedelta(seconds=round(time_taken))
        mean_td = dt.timedelta(seconds=round(mean_time))

        print('Batch {} time: {} (avg: {}, remaining: {})'.format(i, time_td, mean_td, remaining_td))

    print('\nProcessing batches complete. {} apps written'.format(apps_written))

In [65]:
def reset_index(download_path, index_filename):
    """Reset index in file to 0."""
    rel_path = os.path.join(download_path, index_filename)

    with open(rel_path, 'w') as f:
        print(0, file=f)


def get_index(download_path, index_filename):
    """Retrieve index from file, returning 0 if file not found."""
    try:
        rel_path = os.path.join(download_path, index_filename)

        with open(rel_path, 'r') as f:
            index = int(f.readline())

    except FileNotFoundError:
        index = 0

    return index


def prepare_data_file(download_path, filename, index, columns):
    """Create file and write headers if index is 0."""
    if index == 0:
        rel_path = os.path.join(download_path, filename)

        with open(rel_path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=columns)
            writer.writeheader()

In [72]:
def parse_steam_request(appid, name):
    """Unique parser to handle data from Steam Store API.

    Returns : json formatted data (dict-like)
    """
    url = "http://store.steampowered.com/api/appdetails/"
    parameters = {"appids": appid}

    json_data = get_request(url, parameters=parameters)
    json_app_data = json_data[str(appid)]

    if json_app_data['success']:
        data = json_app_data['data']
    else:
        data = {'name': name, 'steam_appid': appid}

    return data


# Set file parameters
download_path = os.getcwd() #dir_save_files
steam_app_data = 'steam_app_data.csv'
steam_index = 'steam_index.txt'

steam_columns = [
    'type', 'name', 'steam_appid', 'required_age', 'is_free', 'controller_support',
    'dlc', 'detailed_description', 'about_the_game', 'short_description', 'fullgame',
    'supported_languages', 'header_image', 'website', 'pc_requirements', 'mac_requirements',
    'linux_requirements', 'legal_notice', 'drm_notice', 'ext_user_account_notice',
    'developers', 'publishers', 'demos', 'price_overview', 'packages', 'package_groups',
    'platforms', 'metacritic', 'reviews', 'categories', 'genres', 'screenshots',
    'movies', 'recommendations', 'achievements', 'release_date', 'support_info',
    'background', 'content_descriptors'
]

# Overwrites last index for demonstration (would usually store highest index so can continue across sessions)
reset_index(download_path, steam_index)

# Retrieve last index downloaded from file
index = get_index(download_path, steam_index)

# Wipe or create data file and write headers if index is 0
prepare_data_file(download_path, steam_app_data, index, steam_columns)

# Set end and chunksize for demonstration - remove to run through entire app list
process_batches(
    parser=parse_steam_request,
    app_list=app_list,
    download_path=download_path,
    data_filename=steam_app_data,
    index_filename=steam_index,
    columns=steam_columns,
    begin=index,
    end=-1,
    batchsize=50
)


Starting at index 0:

Exported lines 0-49 to steam_app_data.csv. Batch 0 time: 0:02:40 (avg: 0:02:40, remaining: 6 days, 6:24:54)
Exported lines 50-99 to steam_app_data.csv. Batch 1 time: 0:02:40 (avg: 0:02:40, remaining: 6 days, 6:16:40)
Exported lines 100-149 to steam_app_data.csv. Batch 2 time: 0:02:45 (avg: 0:02:42, remaining: 6 days, 7:59:58)
Exported lines 150-199 to steam_app_data.csv. Batch 3 time: 0:02:36 (avg: 0:02:40, remaining: 6 days, 6:42:39)
Exported lines 200-249 to steam_app_data.csv. Batch 4 time: 0:02:37 (avg: 0:02:40, remaining: 6 days, 5:58:35)


ConnectionError: ('Connection aborted.', TimeoutError(10060, 'Se produjo un error durante el intento de conexión ya que la parte conectada no respondió adecuadamente tras un periodo de tiempo, o bien se produjo un error en la conexión establecida ya que el host conectado no ha podido responder', None, 10060, None))

In [71]:
df_download = pd.read_csv('steam_app_data.csv')